# Week 3 Session 6: Tuning a CNN + Meet a Transformer

This session has two halves:

1. **Tune the CNN** from Session 5 -- look inside it, then sweep through hyperparameters and see which ones actually move test accuracy.
2. **Meet a different brain** -- the Vision Transformer (ViT). We use a pre-trained ViT from Hugging Face and run it on the same images, head to head with the CNN.

## Activities in this Session

| # | Activity | Topic | Style |
|---|----------|-------|-------|
| 1 | Inside the CNN | Filter & feature-map visualisation | Worked + 1 TODO |
| 2 | Hyperparameter experiments | Kernel size, filter count, pooling, dropout, depth, custom config | Worked + 3 TODOs |
| 3 | A different brain: ViT | Hugging Face image-classification pipeline, CNN vs ViT | Worked + 1 TODO |

> **Heads up:** the first time we load the ViT pipeline Hugging Face will download ~350 MB. Run the install/import cell early.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10

np.random.seed(42)
tf.random.set_seed(42)

# Reload the CIFAR-10 subset (same recipe as Session 5)
(X_train_full, y_train_full), (X_test_full, y_test_full) = cifar10.load_data()
rng = np.random.default_rng(42)
train_idx = rng.permutation(len(X_train_full))[:10000]
test_idx  = rng.permutation(len(X_test_full))[:2000]

X_train = X_train_full[train_idx].astype('float32') / 255.0
X_test  = X_test_full[test_idx].astype('float32')  / 255.0
y_train = y_train_full[train_idx].flatten()
y_test  = y_test_full[test_idx].flatten()

class_names = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']
print(f'X_train: {X_train.shape}   X_test: {X_test.shape}')

---
# Activity 1 -- Inside the CNN

> ### Activity Card
> **Goal:** Visualise what the CNN actually learned -- the filters in the first conv layer and the feature maps they produce on a real image.
> **Why this matters:** This is the moment a CNN stops being a black box. Filters look like edge / colour / texture detectors. Feature maps show where in the image those patterns live.

In [ ]:
# Try to load the CNN saved at the end of Session 5; otherwise build & train a fresh one
import os
if os.path.exists('week3_cnn.keras'):
    model_cnn = models.load_model('week3_cnn.keras')
    print('Loaded saved CNN from Session 5.')
else:
    print('No saved CNN found. Building and training a fresh one (5 epochs)...')
    model_cnn = models.Sequential([
        layers.Input(shape=(32, 32, 3)),
        layers.Conv2D(32, kernel_size=3, activation='relu', padding='same'),
        layers.MaxPooling2D(pool_size=2),
        layers.Conv2D(64, kernel_size=3, activation='relu', padding='same'),
        layers.MaxPooling2D(pool_size=2),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dense(10, activation='softmax'),
    ])
    model_cnn.compile(optimizer='adam',
                      loss='sparse_categorical_crossentropy',
                      metrics=['accuracy'])
    model_cnn.fit(X_train, y_train, epochs=5, batch_size=64,
                  validation_data=(X_test, y_test), verbose=2)
    model_cnn.save('week3_cnn.keras')

baseline_loss, baseline_acc = model_cnn.evaluate(X_test, y_test, verbose=0)
print(f'\nBaseline CNN test accuracy: {baseline_acc:.4f}')

In [ ]:
# Visualise the filters in the FIRST conv layer
first_conv = model_cnn.layers[0]
filters, biases = first_conv.get_weights()
print(f'filters shape: {filters.shape}   (3x3 kernel, 3 input channels, 32 filters)')

# Normalise filter values to 0-1 for plotting
f_min, f_max = filters.min(), filters.max()
filters_norm = (filters - f_min) / (f_max - f_min)

fig, axes = plt.subplots(4, 8, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(filters_norm[:, :, :, i])  # show all 3 input channels as RGB
    ax.set_title(f'#{i}', fontsize=8)
    ax.axis('off')
plt.suptitle('First conv layer -- 32 learned 3x3 filters')
plt.tight_layout()
plt.show()

### TODO 1.1 -- Visualise the feature maps for one image

A **feature map** is the output of a conv layer for a single image -- one map per filter. Bright pixels mean the filter "fired" at that location. Use `tf.keras.Model(inputs=..., outputs=first_conv.output)` to build a sub-model that returns the first conv layer's output, then plot the first 8 feature maps for one image.

In [ ]:
# Build a sub-model that returns the output of the first conv layer
feature_extractor = tf.keras.Model(inputs=model_cnn.inputs,
                                   outputs=model_cnn.layers[0].output)

# TODO: pick a test image
my_idx = 0
one_image_batch = X_test[my_idx:my_idx+1]   # shape (1, 32, 32, 3)

# TODO: run feature_extractor.predict(one_image_batch) to get feature maps
# Hint: result shape will be (1, 32, 32, 32) -- 32 feature maps of size 32x32


# TODO: plot the first 8 feature maps in a 2x4 grid (use cmap='viridis')
# Hint: feature_maps[0, :, :, i] for i in range(8)


### What do the deeper layers see?
- **First conv layer** -> simple things: edges, colour blobs, basic textures.
- **Deeper conv layers** -> combinations: corners, stripes, eye-like patches, fur-like textures.
- **Last layers** -> whole-object concepts: "cat-like", "wheel-like".

This **hierarchy of features** is the secret behind why CNNs work on images.

### Reflection (Activity 1)
1. Look at the 32 filters in the grid above. Do any of them clearly look like edge or colour detectors?
2. In the feature maps for your chosen image, which filters lit up most strongly? What part of the image were they responding to?

---
# Activity 2 -- Hyperparameter Experiments

> ### Activity Card
> **Goal:** Stop guessing, start measuring. We change one thing at a time and watch the test accuracy.
> **Pattern:** define a small `build_cnn(...)` helper, then for each experiment swap one argument, train for 3 epochs, record the score.
> **Task:** Run the worked experiments, then complete three TODO experiments -- the last one is your own custom config (post your best score in the leaderboard cell at the end).

In [ ]:
# Helper: build a CNN with configurable hyperparameters
def build_cnn(num_filters=32, kernel_size=3, pooling='max',
              dropout=0.0, deeper=False):
    Pool = layers.MaxPooling2D if pooling == 'max' else layers.AveragePooling2D
    m = models.Sequential()
    m.add(layers.Input(shape=(32, 32, 3)))
    m.add(layers.Conv2D(num_filters, kernel_size, activation='relu', padding='same'))
    m.add(Pool(pool_size=2))
    m.add(layers.Conv2D(num_filters * 2, kernel_size, activation='relu', padding='same'))
    m.add(Pool(pool_size=2))
    if deeper:
        m.add(layers.Conv2D(num_filters * 4, kernel_size, activation='relu', padding='same'))
        m.add(Pool(pool_size=2))
    m.add(layers.Flatten())
    m.add(layers.Dense(64, activation='relu'))
    if dropout > 0:
        m.add(layers.Dropout(dropout))
    m.add(layers.Dense(10, activation='softmax'))
    m.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
    return m

def train_and_score(model, epochs=3, label='model'):
    h = model.fit(X_train, y_train, epochs=epochs, batch_size=64,
                  validation_data=(X_test, y_test), verbose=0)
    acc = model.evaluate(X_test, y_test, verbose=0)[1]
    print(f'{label:30s}  test acc = {acc:.4f}')
    return acc, h

# Collect every experiment in this dict so we can plot them at the end
results = {'baseline (k=3, f=32, MaxPool)': baseline_acc}
print('Helper functions ready. baseline acc already recorded.')

### Experiment A -- Kernel size 3 vs 7 (worked)

In [ ]:
# Experiment A: change the kernel size from 3 to 7
acc_k7, _ = train_and_score(build_cnn(kernel_size=7), label='kernel=7')
results['kernel=7'] = acc_k7

### TODO 2.1 -- Filter count: 8 vs 64

Train two CNNs: one with `num_filters=8` (tiny) and one with `num_filters=64` (big). Record the accuracy of each in `results`. More filters = more capacity = more parameters.

In [ ]:
# TODO: build_cnn(num_filters=8), pass to train_and_score with label='filters=8'
# Hint: acc_f8, _ = train_and_score(build_cnn(num_filters=8), label='filters=8')


# TODO: same for num_filters=64


# TODO: store both in the results dict
# Hint: results['filters=8']  = acc_f8


### Experiment C -- MaxPool vs AvgPool (worked)
- **MaxPool** keeps the strongest activation in each 2x2 region. Sharp, contrast-aware.
- **AvgPool** averages instead. Softer, more like a mild blur.

In [ ]:
acc_avg, _ = train_and_score(build_cnn(pooling='avg'), label='AvgPool')
results['AvgPool'] = acc_avg

### TODO 2.2 -- Dropout

Add **dropout = 0.5** before the final dense layer. Dropout randomly switches off half the neurons during training -- it's the cheapest, most reliable way to fight overfitting.

In [ ]:
# TODO: train build_cnn(dropout=0.5) and store as 'dropout=0.5' in results
# Hint: acc_drop, _ = train_and_score(build_cnn(dropout=0.5), label='dropout=0.5')


### Experiment E -- Deeper network (worked)
Add a third Conv -> Pool block. More depth often helps, but only up to a point.

In [ ]:
acc_deep, _ = train_and_score(build_cnn(deeper=True), label='deeper (3 conv blocks)')
results['deeper'] = acc_deep

### TODO 2.3 -- Your own custom config (LEADERBOARD!)

Pick any combination of the hyperparameters you have just seen -- kernel size, filter count, pooling, dropout, depth -- and try to beat the baseline. Post your best test accuracy in the markdown cell below.

In [ ]:
# TODO: build YOUR custom CNN. Some suggestions:
#   build_cnn(num_filters=64, kernel_size=5, dropout=0.3, deeper=True)
#   build_cnn(num_filters=48, pooling='max', dropout=0.4, deeper=True)
# Tweak until you like the score, then add it to results.

# my_acc, _ = train_and_score(build_cnn(...), label='my custom config')
# results['my custom config'] = my_acc


### Class Leaderboard
Drop your best score here so we can compare:

| Student | Config | Test Accuracy |
|---------|--------|---------------|
| _name_  | _e.g. f=64, k=5, dropout=0.3, deeper_ | _0.xxxx_ |
| _name_  |        |               |
| _name_  |        |               |

In [ ]:
# Summary bar chart of every experiment we ran
labels = list(results.keys())
scores = list(results.values())

plt.figure(figsize=(10, 4))
bars = plt.bar(range(len(labels)), scores, color='steelblue', edgecolor='black')
plt.xticks(range(len(labels)), labels, rotation=30, ha='right')
plt.ylabel('test accuracy')
plt.ylim(0, max(scores) + 0.05)
plt.title('CNN hyperparameter sweep -- CIFAR-10 (3 epochs each)')
for b, s in zip(bars, scores):
    plt.text(b.get_x() + b.get_width()/2, s + 0.005, f'{s:.3f}',
             ha='center', fontsize=9)
plt.tight_layout()
plt.show()

### Lessons from the sweep
- **More capacity is not always better.** Bigger filter counts and deeper networks can overfit on a 10k-image subset.
- **Dropout is cheap insurance.** It costs nothing at inference time and usually helps test accuracy.
- **Kernel size matters less than you think** once the network is reasonable -- 3x3 is the safe default.

---
# Activity 3 -- A Different Brain: Vision Transformer (ViT)

> ### Activity Card
> **Goal:** Run a state-of-the-art image model that does **not** use convolution at all -- the Vision Transformer -- and compare it head to head with our CNN on the very same images.
> **The trick:** we don't train it. We use a **pre-trained** ViT from Hugging Face that already knows ImageNet.

### What is a Transformer (in one paragraph)?
A transformer chops the image into small **patches** (here 16x16), turns each patch into a vector, and then asks a single question over and over: *"how much should each patch pay attention to every other patch?"* That mechanism is called **self-attention**, and it replaces convolution entirely.

Important: ViT was pre-trained on ImageNet, so its 1,000 class names are different from CIFAR-10's 10. We will see "tabby cat" instead of "cat", "sports car" instead of "automobile". That is normal -- the goal is to compare top-1 predictions side by side, not class IDs.

In [ ]:
# Make sure the transformers library is installed (it ships with the course env)
try:
    from transformers import pipeline
    print('transformers is installed.')
except ImportError:
    print('Installing transformers...')
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers'])
    from transformers import pipeline

In [ ]:
# Load a pre-trained Vision Transformer (~350 MB download the first time)
from PIL import Image

vit = pipeline('image-classification',
               model='google/vit-base-patch16-224',
               top_k=3)
print('ViT pipeline ready.')

In [ ]:
# Run ViT on 5 sample test images
sample_indices = [0, 1, 2, 3, 4]

vit_predictions = []
for idx in sample_indices:
    # Convert the 32x32 numpy image into a PIL image (ViT expects PIL or a path)
    img = (X_test[idx] * 255).astype('uint8')
    pil_img = Image.fromarray(img).resize((224, 224))   # ViT was trained at 224x224
    preds = vit(pil_img)
    vit_predictions.append(preds)

# Run the CNN on the same 5 images
cnn_probs = model_cnn.predict(X_test[sample_indices], verbose=0)
cnn_top1  = cnn_probs.argmax(axis=1)

# Show the side-by-side
fig, axes = plt.subplots(1, 5, figsize=(15, 4))
for ax, idx, vit_pred, cnn_idx in zip(axes, sample_indices, vit_predictions, cnn_top1):
    ax.imshow(X_test[idx])
    actual = class_names[y_test[idx]]
    cnn_label = class_names[cnn_idx]
    vit_label = vit_pred[0]['label']
    ax.set_title(f"actual: {actual}\nCNN: {cnn_label}\nViT: {vit_label}", fontsize=8)
    ax.axis('off')
plt.suptitle('CNN vs ViT  (top-1 predictions)')
plt.tight_layout()
plt.show()

### TODO 3.1 -- Run ViT on YOUR image from Session 5

Pick the same `my_idx` test image you predicted on in Session 5 Activity 3.2, run ViT on it, and print **both** the CNN's top-3 and ViT's top-3 side by side.

In [ ]:
# TODO: pick the same image index you used in Session 5 TODO 3.2
my_idx = 42

# TODO: convert X_test[my_idx] to a PIL image at 224x224 (see the cell above)


# TODO: get vit predictions on that PIL image (will be a list of dicts with 'label' and 'score')


# TODO: get cnn predictions: probs = model_cnn.predict(X_test[my_idx:my_idx+1])[0]
# Then top3_cnn = probs.argsort()[-3:][::-1]


# TODO: show the image and print both rankings side by side


### Discussion Questions (Activity 3)

1. On the 5 sample images, did the CNN and the ViT ever disagree on the class?
2. ViT was pre-trained on millions of ImageNet images. Our CNN was trained on 10,000. How much of the gap (in either direction) do you think comes from the **architecture** vs the **training data**?
3. ViT gives confident-sounding labels like "tabby cat" or "sports car" because of its training. When would you **want** that detail? When would the CNN's coarser 10-class output be more useful?

---

## When Should You Reach for a Transformer?

| Pick a CNN if... | Pick a transformer if... |
|------------------|---------------------------|
| Your dataset is small (< 50k images) | You can use a large pre-trained model |
| You need fast inference on CPU | You need state-of-the-art accuracy |
| You want full training control | You are happy to fine-tune a pre-trained model |

CNNs are still the workhorse for most "small-data" image problems. Transformers shine when you can stand on the shoulders of a giant pre-trained model.

---

## Wrap-up: Week 3 in 3 bullets

1. **Forward propagation** = input -> layer -> layer -> output. **Back-propagation** = the gradient that says how to adjust every weight. **Gradient descent** = take a small step in that direction. Keras and TensorFlow do all of this for you.
2. **Convolution + pooling** turns raw pixels into a hierarchy of features. That is why a CNN beats a dense network on images at a fraction of the parameter count.
3. **Hyperparameters matter, but not all equally.** Dropout, depth, and filter count moved the needle most in our sweep. Kernel size barely did.

Next week: final-project deep dives.